# 04 — Naive RAG from First Principles

**First Finance - Arnaud Demes**  
**Session:** Day 1, 11:30–12:00 · 8 minutes concepts + 22 minutes notebook

Build the smallest retrieval-augmented generation pipeline you can inspect end to
end. It works, but it is deliberately **naive**: the passages are prepared in
advance, retrieval is lexical TF-IDF, and selection is a simple top-k rule.

## Learning objectives

By the end of this notebook, you can:

1. explain RAG as retrieval plus controlled context construction;
2. build a TF-IDF index and cosine-similarity ranking;
3. visualize why a passage is selected;
4. test retrieval separately from answer generation;
5. identify which naive components parsing, chunking and advanced retrieval improve.

> **Deliverable:** one NVIDIA answer grounded in retrieved evidence and one visible
> failure that motivates Lessons 5 and 6.

## Before you start

- The tested baseline uses a deterministic offline response and needs no API key.
- For Ollama, keep the default provider and ensure the configured model is running.
- For OpenAI, set `FINAI_MODEL_PROVIDER=openai` and `OPENAI_API_KEY` outside the notebook.
- Every chart is calculated from the corpus, query, ranking or prompt built below.

This lesson starts with **prepared passages**. That is a teaching simplification,
not a production ingestion pipeline.

## Where this fits

```text
Lesson 3                       Lesson 4                       Lesson 5
full context stops fitting → naive retrieval baseline → parsing + chunking laboratory
                                                                    ↓
                                                        Lesson 6: better retrieval
```

Lesson 4 isolates the retrieval mechanism. Lesson 5 will improve the source
representation; Lesson 6 will improve how evidence is searched and ranked.

The experiment is cumulative: **No context → full context → naive RAG**. The
question stays fixed while the application changes what evidence the model can see.

## Set up the lab

The implementation uses the same provider-neutral gateway as the previous lessons.
The retrieval calculations do not call a model.

In [ ]:
from __future__ import annotations

import os
from textwrap import shorten

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch

from finai_academy.context import estimate_tokens
from finai_academy.lesson_support import RecordedRagModel
from finai_academy.providers import create_chat_model, provider_summary
from finai_academy.retrieval import (
    EvidencePassage,
    LexicalRetriever,
    build_rag_prompt,
    evaluate_retrieval,
)
from finai_academy.settings import Settings

NAVY = "#051C2A"
ROYAL = "#1F40CB"
CYAN = "#00A2EB"
ORANGE = "#F07D00"
RED = "#D95D5D"
SLATE = "#627D98"
LIGHT = "#D9E2EC"
OFF_WHITE = "#F5F5F5"

np.random.seed(7)
plt.rcParams.update({
    "figure.figsize": (10, 4.8),
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "font.size": 10,
})

## 1. Inspect the prepared financial corpus

The six passages are paraphrased from official disclosures. Provenance travels with
each passage so the final prompt can preserve company, period, section and URL.

- [NVIDIA fiscal 2026 Form 10-K](https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm)
- [Schneider Electric 2025 full-year results](https://www.se.com/ww/en/assets/564/document/528237/release-fy-results-2025.pdf?p_File_Name=2025+FY+Results&p_enDocType=Financial+release)

> The baseline also rebuilds one joined teaching document and applies a **naive paragraph split**.
> In Lesson 5, ready-made passages disappear: you will parse raw financial documents,
> preserve structure and compare stronger chunking strategies.

In [ ]:
NVIDIA_SOURCE = (
    "https://www.sec.gov/Archives/edgar/data/1045810/"
    "000104581026000021/nvda-20260125.htm"
)
SCHNEIDER_SOURCE = (
    "https://www.se.com/ww/en/assets/564/document/528237/"
    "release-fy-results-2025.pdf?p_File_Name=2025+FY+Results&"
    "p_enDocType=Financial+release"
)

passages = (
    EvidencePassage(
        passage_id="NVDA-F1",
        company="NVIDIA",
        period="FY2026",
        section="Results of operations",
        text=(
            "For fiscal 2026, total revenue was $215.938 billion, compared with "
            "$130.497 billion in fiscal 2025. NVIDIA reported 65% year-on-year growth."
        ),
        source_url=NVIDIA_SOURCE,
    ),
    EvidencePassage(
        passage_id="NVDA-F2",
        company="NVIDIA",
        period="FY2026",
        section="Revenue by end market",
        text=(
            "Data Center revenue was $193.737 billion in fiscal 2026, up 68% from "
            "fiscal 2025, with growth driven by accelerated computing and AI demand."
        ),
        source_url=NVIDIA_SOURCE,
    ),
    EvidencePassage(
        passage_id="NVDA-F3",
        company="NVIDIA",
        period="FY2026",
        section="Gaming",
        text=(
            "Gaming revenue was $16.042 billion in fiscal 2026, up 41% from fiscal "
            "2025, with growth driven by demand for Blackwell products."
        ),
        source_url=NVIDIA_SOURCE,
    ),
    EvidencePassage(
        passage_id="SU-F1",
        company="Schneider Electric",
        period="FY2025",
        section="Group results",
        text=(
            "Schneider Electric reported fiscal 2025 revenue of €40.152 billion, "
            "with organic growth of 8.9%."
        ),
        source_url=SCHNEIDER_SOURCE,
    ),
    EvidencePassage(
        passage_id="SU-F2",
        company="Schneider Electric",
        period="FY2025",
        section="Energy Management",
        text=(
            "Energy Management organic revenue increased 10% in fiscal 2025. "
            "Data Center demand led Energy Management growth in the fourth quarter."
        ),
        source_url=SCHNEIDER_SOURCE,
    ),
    EvidencePassage(
        passage_id="SU-F3",
        company="Schneider Electric",
        period="FY2025",
        section="Profitability",
        text=(
            "Adjusted EBITA was €7.520 billion in fiscal 2025 at an 18.7% margin, "
            "representing 12.3% organic growth."
        ),
        source_url=SCHNEIDER_SOURCE,
    ),
)

prepared_document = "\n\n".join(passage.text for passage in passages)
naive_paragraphs = tuple(
    paragraph.strip()
    for paragraph in prepared_document.split("\n\n")
    if paragraph.strip()
)

NO_CONTEXT_REFERENCE = (
    "Insufficient evidence: the question names no supplied financial source."
)
FULL_CONTEXT_REFERENCE = (
    "NVIDIA fiscal 2026 Data Center revenue was $193.7 billion [NVDA-F2] "
    "within total revenue of $215.9 billion [NVDA-F1]. The complete corpus also "
    "contains Schneider evidence, so the answer must keep company boundaries explicit."
)
baseline_reference_table = pd.DataFrame([
    {"path": "no_context", "evidence_visible": 0, "recorded_result": NO_CONTEXT_REFERENCE},
    {"path": "full_context", "evidence_visible": len(passages), "recorded_result": FULL_CONTEXT_REFERENCE},
])

print("naive paragraph split:", len(naive_paragraphs), "paragraphs")
print("Baseline references before retrieval:")
print(baseline_reference_table.to_string(index=False))

corpus_table = pd.DataFrame([
    {
        "id": passage.passage_id,
        "company": passage.company,
        "period": passage.period,
        "section": passage.section,
        "estimated_tokens": estimate_tokens(passage.text),
        "preview": shorten(passage.text, width=82, placeholder="…"),
    }
    for passage in passages
])
corpus_table

### Visual 1 — A corpus is a set of evidence units

Each bar is one prepared passage. The model will not see all six: the retriever will
decide which evidence enters the prompt.

In [ ]:
plot_corpus = corpus_table.sort_values(["company", "id"], ascending=[False, True])
company_colors = {
    "NVIDIA": ROYAL,
    "Schneider Electric": ORANGE,
}

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.barh(
    plot_corpus["id"],
    plot_corpus["estimated_tokens"],
    color=[company_colors[company] for company in plot_corpus["company"]],
    alpha=0.9,
)
ax.bar_label(bars, labels=plot_corpus["section"], padding=5, fontsize=9)
ax.set_xlim(0, plot_corpus["estimated_tokens"].max() * 1.55)
ax.set_title("Prepared NVIDIA–Schneider evidence corpus")
ax.set_xlabel("Estimated tokens per passage")
ax.set_ylabel("Stable passage identifier")
ax.spines[["top", "right"]].set_visible(False)
ax.text(0.99, 0.03, "Blue = NVIDIA · Orange = Schneider Electric", transform=ax.transAxes, ha="right", color=SLATE)
plt.tight_layout()
plt.show()

## 2. Build the lexical index

TF-IDF assigns more weight to terms that are frequent in one passage but less common
across the corpus. Cosine similarity then compares the direction of the query vector
with each passage vector.

The implementation fits once on the corpus. No language model is involved.

In [ ]:
retriever = LexicalRetriever(passages)
QUESTION = "Which NVIDIA business drove fiscal 2026 total revenue and Data Center growth?"
TOP_K = 2

query_weights = retriever.query_weights(QUESTION)
query_terms = {
    term for term in QUESTION.casefold().replace("?", "").split()
    if term in retriever.feature_names
}
focus_terms = [
    term for term in retriever.feature_names
    if term in query_terms or term in {"data", "center", "gaming", "management"}
]

print(f"Corpus passages: {len(passages)}")
print(f"Vocabulary terms: {len(retriever.feature_names)}")
print("Question terms represented in the index:", sorted(query_terms))

### Visual 2 — See the representation before seeing the ranking

The heatmap shows document TF-IDF weights for selected terms plus the query vector.
Dark cells are the lexical overlap the similarity calculation can use.

In [ ]:
term_positions = [retriever.feature_names.index(term) for term in focus_terms]
teaching_matrix = np.vstack([
    retriever.document_term_matrix[:, term_positions],
    query_weights[term_positions],
])
row_labels = [passage.passage_id for passage in passages] + ["QUERY"]

fig, ax = plt.subplots(figsize=(10, 5.2))
image = ax.imshow(teaching_matrix, cmap="Blues", aspect="auto", vmin=0)
ax.set_xticks(range(len(focus_terms)), labels=focus_terms, rotation=40, ha="right")
ax.set_yticks(range(len(row_labels)), labels=row_labels)
ax.axhline(len(passages) - 0.5, color=ORANGE, linewidth=2)
ax.set_title("TF-IDF weights: selected corpus terms and the query")
ax.set_xlabel("Vocabulary terms retained for explanation")
ax.set_ylabel("Passages and query")
fig.colorbar(image, ax=ax, label="TF-IDF weight", fraction=0.03, pad=0.03)
plt.tight_layout()
plt.show()

## 3. Rank every passage, then apply top-k

Retrieval is a ranking problem. `top_k=2` is an application decision: increasing it
may recover more relevant evidence, but also adds noise, tokens and cost.

In [ ]:
ranking = retriever.rank(QUESTION)
hits = retriever.search(QUESTION, top_k=TOP_K)
selected_ids = {hit.passage.passage_id for hit in hits}

ranking_table = pd.DataFrame([
    {
        "rank": rank,
        "id": item.passage.passage_id,
        "company": item.passage.company,
        "section": item.passage.section,
        "score": item.score,
        "selected": item.passage.passage_id in selected_ids,
    }
    for rank, item in enumerate(ranking, start=1)
])
ranking_table

### Visual 3 — The top-k boundary is visible

Orange bars cross the selection boundary. Grey passages stay outside the model
context even though they remain in the corpus.

In [ ]:
rank_plot = ranking_table.sort_values("score")
colors = [ORANGE if selected else LIGHT for selected in rank_plot["selected"]]

fig, ax = plt.subplots(figsize=(10, 4.6))
bars = ax.barh(rank_plot["id"], rank_plot["score"], color=colors, edgecolor=NAVY, linewidth=0.5)
ax.bar_label(bars, labels=[f"{score:.3f}" for score in rank_plot["score"]], padding=4)
ax.set_xlim(0, max(rank_plot["score"].max() * 1.25, 0.1))
ax.set_title(f"Cosine-similarity ranking · top_k={TOP_K}")
ax.set_xlabel("Cosine similarity to the question")
ax.set_ylabel("Evidence passage")
ax.spines[["top", "right"]].set_visible(False)
ax.text(0.99, 0.03, "Orange = sent to the model", transform=ax.transAxes, ha="right", color=SLATE)
plt.tight_layout()
plt.show()

print("Selected evidence:", [hit.passage.passage_id for hit in hits])

## 4. Assemble controlled context

RAG does not give the model access to a database. It constructs a new prompt from the
retrieved passages. Stable identifiers and source URLs must survive this step.

In [ ]:
rag_prompt = build_rag_prompt(QUESTION, hits)
prompt_components = {
    "Instructions + markup": max(
        estimate_tokens(rag_prompt) - estimate_tokens(QUESTION) - sum(estimate_tokens(hit.passage.text) for hit in hits),
        0,
    ),
    **{
        hit.passage.passage_id: estimate_tokens(hit.passage.text)
        for hit in hits
    },
    "Question": estimate_tokens(QUESTION),
}

print(rag_prompt)

### Visual 4 — Retrieval controls the prompt budget

The complete corpus remains outside the model call. Only instructions, selected
evidence and the question consume input capacity.

In [ ]:
full_corpus_tokens = sum(estimate_tokens(passage.text) for passage in passages)
selected_evidence_tokens = sum(estimate_tokens(hit.passage.text) for hit in hits)

fig, (ax_selection, ax_prompt) = plt.subplots(1, 2, figsize=(12, 4.2))

selection_bars = ax_selection.barh(
    ["Complete corpus evidence", "Selected top-k evidence"],
    [full_corpus_tokens, selected_evidence_tokens],
    color=[LIGHT, ORANGE],
    edgecolor=NAVY,
    linewidth=0.6,
)
ax_selection.bar_label(selection_bars, labels=[f"~{full_corpus_tokens}", f"~{selected_evidence_tokens}"], padding=4)
ax_selection.set_title("Evidence selection")
ax_selection.set_xlabel("Estimated evidence tokens")
ax_selection.spines[["top", "right"]].set_visible(False)

left = 0
component_colors = [NAVY, ROYAL, CYAN, ORANGE]
for (label, value), color in zip(prompt_components.items(), component_colors, strict=True):
    ax_prompt.barh([0], [value], left=left, color=color, label=f"{label}: ~{value}")
    if value >= 12:
        ax_prompt.text(left + value / 2, 0, f"~{value}", ha="center", va="center", color="white")
    left += value

ax_prompt.set_yticks([])
ax_prompt.set_title("Final model input after retrieval")
ax_prompt.set_xlabel("Estimated prompt tokens")
ax_prompt.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.25), frameon=False, fontsize=9)
ax_prompt.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Generate from the retrieved evidence

Only this cell crosses the model boundary. Retrieval remains deterministic and can be
tested even when no model is available.

In [ ]:
settings = Settings.from_environment()
live_mode = os.getenv("FINAI_LIVE_MODE", "1") == "1"
model = create_chat_model(settings) if live_mode else RecordedRagModel()

response = model.invoke([
    ("system", "Use only the retrieved evidence and preserve passage citations."),
    ("human", rag_prompt),
])
answer = str(response.content)
path_comparison = pd.DataFrame([
    {"path": "no_context", "evidence_visible": 0, "result": NO_CONTEXT_REFERENCE},
    {"path": "full_context", "evidence_visible": len(passages), "result": FULL_CONTEXT_REFERENCE},
    {"path": "naive_rag", "evidence_visible": len(hits), "result": answer},
])

print("Execution mode:", "live" if live_mode else "offline fixture")
print("Provider configuration:", provider_summary(settings))
print("\nANSWER\n", answer)
print("\nPath comparison:")
print(path_comparison[["path", "evidence_visible"]].to_string(index=False))

## Verification

Retrieval and generation answer different questions:

- **Retrieval check:** did the prompt receive the expected evidence?
- **Grounding check:** did the answer use that evidence and preserve limitations?

A fluent answer cannot compensate for missing passages. **Live answer grounding remains an observation**:
the offline recorded answer must pass all five checks, while an Ollama/OpenAI REVIEW
remains visible without invalidating the separately verified retrieval pipeline.

In [ ]:
retrieval_check = evaluate_retrieval(hits, {"NVDA-F1", "NVDA-F2"})
grounding_checks = {
    "identifies Data Center": "data center" in answer.casefold(),
    "uses the Data Center metric": "193.7" in answer,
    "uses the total revenue metric": "215.9" in answer,
    "cites both retrieved passages": all(identifier in answer for identifier in ("[NVDA-F1]", "[NVDA-F2]")),
    "states an evidence limitation": "does not establish" in answer.casefold(),
}

print(f"Retrieval check: recall={retrieval_check.recall:.0%} · missing={retrieval_check.missing_ids}")
for criterion, passed in grounding_checks.items():
    print(f"{'PASS' if passed else 'REVIEW':6} {criterion}")
print(f"Grounding check: {sum(grounding_checks.values())}/{len(grounding_checks)}")

## Failure lab

TF-IDF only sees lexical overlap. Ask the same financial question using different
language and inspect the ranking again:

> Which division supplied most of the expansion from machine-learning infrastructure?

The relevant idea is still Data Center, but the query avoids most of the source
vocabulary. This is a controlled representation failure, not a model failure.

In [ ]:
FAILURE_QUESTION = (
    "Which division supplied most of the expansion from machine-learning infrastructure?"
)
failure_ranking = retriever.rank(FAILURE_QUESTION)

comparison = pd.DataFrame({
    "id": [item.passage.passage_id for item in ranking],
    "baseline_score": [item.score for item in ranking],
}).merge(
    pd.DataFrame({
        "id": [item.passage.passage_id for item in failure_ranking],
        "failure_score": [item.score for item in failure_ranking],
    }),
    on="id",
)
comparison

### Visual 5 — Vocabulary changes can reorder the evidence

The paired chart compares scores for the explicit query and the semantically related
failure query. Zero lexical overlap creates ties; stable identifiers then determine
the order, not financial relevance.

In [ ]:
comparison = comparison.sort_values("baseline_score", ascending=True)
y = np.arange(len(comparison))

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.barh(y - 0.17, comparison["baseline_score"], height=0.32, color=ROYAL, label="Explicit vocabulary")
ax.barh(y + 0.17, comparison["failure_score"], height=0.32, color=ORANGE, label="Lexical mismatch")
ax.set_yticks(y, labels=comparison["id"])
ax.set_title("Same intent, different vocabulary, different retrieval signal")
ax.set_xlabel("Cosine similarity")
ax.set_ylabel("Evidence passage")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

failure_top_one = retriever.search(FAILURE_QUESTION, top_k=1)[0]
print("Failure query top-1:", failure_top_one.passage.passage_id)
print("Expected evidence for the intent: NVDA-F2")

### Visual 6 — Every naive component has a specific upgrade

Do not conclude that “RAG works” or “RAG fails” as one undifferentiated system.
Improve the component that produced the measured weakness.

In [ ]:
improvements = [
    ("Prepared passages", "Raw parsing + structure", "Lesson 5"),
    ("Naive boundaries", "Chunking strategies + metadata", "Lesson 5"),
    ("Lexical TF-IDF", "Embeddings + hybrid retrieval", "Lesson 6"),
    ("Simple top-k", "Filtering + reranking", "Lesson 6"),
]

fig, ax = plt.subplots(figsize=(11, 5.2))
ax.set_xlim(0, 10)
ax.set_ylim(0, len(improvements) + 1)
ax.axis("off")
ax.text(2.0, len(improvements) + 0.55, "NAIVE BASELINE", ha="center", color=SLATE, weight="bold")
ax.text(7.3, len(improvements) + 0.55, "MEASURED UPGRADE", ha="center", color=SLATE, weight="bold")

for row, (baseline, upgrade, lesson) in enumerate(improvements, start=1):
    y_pos = len(improvements) + 0.3 - row
    left_box = FancyBboxPatch((0.5, y_pos), 3.0, 0.62, boxstyle="round,pad=0.05", facecolor=OFF_WHITE, edgecolor=NAVY, linewidth=1.2)
    right_box = FancyBboxPatch((5.2, y_pos), 4.2, 0.62, boxstyle="round,pad=0.05", facecolor="#E8F6FB", edgecolor=CYAN, linewidth=1.2)
    ax.add_patch(left_box)
    ax.add_patch(right_box)
    ax.text(2.0, y_pos + 0.31, baseline, ha="center", va="center", color=NAVY, weight="bold")
    ax.text(7.0, y_pos + 0.31, upgrade, ha="center", va="center", color=NAVY, weight="bold")
    ax.annotate("", xy=(5.05, y_pos + 0.31), xytext=(3.65, y_pos + 0.31), arrowprops={"arrowstyle": "->", "color": ORANGE, "lw": 2})
    ax.text(9.65, y_pos + 0.31, lesson, va="center", color=ORANGE, weight="bold")

ax.set_title("The naive RAG baseline creates the improvement roadmap", pad=18)
plt.tight_layout()
plt.show()

In [ ]:
verification = {
    "expected evidence retrieved": retrieval_check.passed,
    "lexical failure is visible": failure_top_one.passage.passage_id != "NVDA-F2",
    "prompt contains source provenance": NVIDIA_SOURCE in rag_prompt,
    "naive paragraph split is visible": len(naive_paragraphs) == len(passages),
    "three context paths compared": tuple(path_comparison["path"]) == (
        "no_context", "full_context", "naive_rag"
    ),
}

for criterion, passed in verification.items():
    print(f"{'PASS' if passed else 'REVIEW':6} {criterion}")

if live_mode:
    print("Live grounding observation:", "PASS" if all(grounding_checks.values()) else "REVIEW")
if not live_mode:
    assert all(grounding_checks.values()), "The recorded RAG answer must pass grounding."

assert all(verification.values()), "Review the visible naive RAG checks before continuing."
print("PASS — naive RAG baseline verified")

## Challenge

Change `TOP_K` from 2 to 1 and rerun the ranking, prompt and verification cells.

1. Which expected passage disappears?
2. Does the answer still have enough evidence to compare total revenue with Data Center?
3. Increase `TOP_K` to 4. Which irrelevant or cross-company passages enter the prompt?
4. Write one sentence choosing a top-k policy for this question.

**Expected reasoning:** top-k=1 loses evidence coverage; top-k=4 adds unnecessary
context. The correct policy depends on the question and must be evaluated.

### Knowledge check

1. Which path exposes the whole mixed-company corpus?
2. Which path creates a ranking before generation?
3. Why is the paragraph split called naive?

**Answers:** full context exposes all six passages; naive RAG ranks before the model
call; blank-line boundaries ignore headings, tables, semantics and hierarchy.

## Troubleshooting

| Symptom | Likely cause | Action |
|---|---|---|
| all similarity scores are zero | query vocabulary is absent from the prepared corpus | inspect the heatmap; do not blame the model |
| relevant passage is below top-k | lexical mismatch or weak passage representation | record the retrieval failure for Lessons 5–6 |
| answer lacks citations | prompt contract or provider output failed | inspect the assembled prompt and rerun the model cell |
| Ollama connection refused | local server is not running | start Ollama and confirm the configured model exists |
| OpenAI authentication error | missing or invalid external key | set `OPENAI_API_KEY` outside the notebook |

## Capstone integration

The Financial Analyst Copilot now has its first retrieval-backed path:

```text
question → naive lexical retrieval → labelled evidence → model → grounded answer
```

Keep the passage identifiers, similarity scores, selected top-k, source URLs and
verification result with each run. These become trace data in the evaluation lesson.

## Recap

- RAG is retrieval plus controlled context construction.
- Retrieval quality and generation quality must be tested separately.
- TF-IDF is a useful transparent baseline, not a semantic understanding system.
- Prepared passages hide parsing and chunking decisions.
- Lesson 5 improves how financial documents become evidence units.
- Lesson 6 improves how those units are represented, filtered and ranked.